# 量价分布与流动性因子 v3

基于1分钟K线的日频OHLCV聚合（含成交量/成交额统计），计算13个量价子因子后等权合成。方向已修复。⭐当前最强因子。

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # 量价分布与流动性因子 (Volume-Price Distribution & Liquidity) v3
# 
# ## 因子逻辑
# 
# ### 经济学直觉
# 量价关系包含丰富的市场微观结构信息：
# 1. **VWAP偏离**：收盘价相对VWAP反映日内的买卖力量平衡
# 2. **价格形态**：振幅、收盘位置、影线长度反映多空博弈结果
# 3. **成交量特征**：量的变异系数和峰值度反映信息冲击
# 4. **成交额特征**：金额的波动反映大资金活动
# 
# ### 子因子（10个）
# - VWAP偏离、开盘vs VWAP、均价偏离
# - 日内收益、价格路径效率、收盘位置
# - 振幅、量变异系数、量峰值度、额变异系数
# 
# ### 改进（v3）
# - SQL只做基础OHLCV聚合，不依赖m_lag
# - 所有衍生因子在pandas中计算

# In[ ]:


def main(datasources, start_date, end_date):
    """
    量价分布与流动性因子 (v3)
    
    SQL只做基础OHLCV聚合，所有衍生因子在pandas中计算。
    因子方向：值越大代表买方力量越强，预期正相关。
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import structlog

    logger = structlog.get_logger()
    t_total = time.time()

    bar1m_table = datasources['bar1m']

    # 时序因子需要向前多取数据用于滚动计算（参考baseline模板）
    lookback_start = pd.to_datetime(start_date) - pd.Timedelta(days=30)

    # ============================================================
    # 第1步：SQL基础日频聚合
    # ============================================================
    logger.info("Step 1: SQL基础日频OHLCV聚合")
    t1 = time.time()

    sql = f"""
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument,
        first(open) AS open_p,
        last(close) AS close_p,
        first(pre_close) AS pre_close_p,
        MAX(high) AS high_p,
        MIN(low) AS low_p,
        SUM(volume) AS total_volume,
        SUM(amount) AS total_amount,
        SUM(close * volume) / NULLIF(SUM(volume), 0) AS vwap,
        AVG(volume) AS avg_vol,
        STDDEV(volume) AS std_vol,
        MAX(volume) AS max_vol,
        AVG(amount) AS avg_amt,
        STDDEV(amount) AS std_amt,
        MAX(amount) AS max_amt,
        COUNT(*) AS minute_count
    FROM {bar1m_table}
    GROUP BY date::DATE::DATETIME, instrument
    ORDER BY trading_day, instrument
    """

    daily = dai.query(sql, filters={'date': [lookback_start, end_date]}, compression=True).df()
    daily['trading_day'] = pd.to_datetime(daily['trading_day'])
    daily['instrument'] = daily['instrument'].astype(str)

    # 将DAI返回的Decimal列转为float，避免和Python float运算时报错
    numeric_cols = daily.select_dtypes(include=['object', 'number']).columns
    for col in numeric_cols:
        if col not in ('date', 'instrument'):
            daily[col] = pd.to_numeric(daily[col], errors='coerce')

    logger.info(f"  -> 日频聚合完成: {len(daily)} 行, {daily['instrument'].nunique()} 标的, "
                f"日期 [{daily['trading_day'].min().date()}, {daily['trading_day'].max().date()}], "
                f"耗时 {time.time()-t1:.1f}s")

    # ============================================================
    # 第2步：pandas计算所有衍生子因子
    # ============================================================
    logger.info("Step 2: pandas计算衍生子因子")
    t2 = time.time()
    eps = 1e-8

    # --- 价格类 ---
    daily['overnight_gap'] = (daily['open_p'] - daily['pre_close_p']) / (daily['pre_close_p'] + eps)
    daily['intraday_return'] = (daily['close_p'] - daily['open_p']) / (daily['open_p'] + eps)
    daily['daily_return'] = (daily['close_p'] - daily['pre_close_p']) / (daily['pre_close_p'] + eps)
    daily['vwap_deviation'] = (daily['close_p'] - daily['vwap']) / (daily['vwap'] + eps)
    daily['open_vs_vwap'] = (daily['open_p'] - daily['vwap']) / (daily['vwap'] + eps)
    daily['price_efficiency'] = (daily['close_p'] - daily['open_p']) / (daily['high_p'] - daily['low_p'] + eps)
    daily['close_position'] = (daily['close_p'] - daily['low_p']) / (daily['high_p'] - daily['low_p'] + eps)
    daily['amplitude'] = (daily['high_p'] - daily['low_p']) / (daily['pre_close_p'] + eps)

    # --- 上/下影线 ---
    body_top = daily[['open_p', 'close_p']].max(axis=1)
    body_bot = daily[['open_p', 'close_p']].min(axis=1)
    daily['upper_shadow'] = (daily['high_p'] - body_top) / (daily['high_p'] - daily['low_p'] + eps)
    daily['lower_shadow'] = (body_bot - daily['low_p']) / (daily['high_p'] - daily['low_p'] + eps)

    # --- 均价偏离 ---
    daily['avg_price'] = daily['total_amount'] / (daily['total_volume'] + eps)
    daily['avg_price_dev'] = (daily['close_p'] - daily['avg_price']) / (daily['avg_price'] + eps)

    # --- 量价类 ---
    daily['vol_cv'] = daily['std_vol'] / (daily['avg_vol'] + eps)          # 量变异系数
    daily['vol_max_ratio'] = daily['max_vol'] / (daily['avg_vol'] + eps)   # 量峰值度
    daily['amt_cv'] = daily['std_amt'] / (daily['avg_amt'] + eps)          # 额变异系数
    daily['amt_max_ratio'] = daily['max_amt'] / (daily['avg_amt'] + eps)   # 额峰值度
    
    # --- 成交量趋势（今日总量 vs 5日均量）---
    daily['vol_ma5'] = daily.groupby('instrument')['total_volume'].transform(
        lambda x: x.rolling(5, min_periods=3).mean()
    )
    daily['vol_ratio_vs_ma5'] = daily['total_volume'] / (daily['vol_ma5'] + eps)

    logger.info(f"  -> 子因子计算完成, 耗时 {time.time()-t2:.1f}s")

    # ============================================================
    # 第3步：极端值处理 + 时序标准化 + 合成
    # ============================================================
    logger.info("Step 3: 极端值处理 + 时序标准化 + 合成")
    t3 = time.time()

    # 子因子选择
    sub_factors = [
        'overnight_gap',
        'intraday_return',
        'daily_return',
        'vwap_deviation',
        'open_vs_vwap',
        'price_efficiency',
        'close_position',
        'upper_shadow',          # 上影线长=抛压，取负
        'avg_price_dev',
        'vol_cv',                # 量波动大=信息冲击
        'vol_max_ratio',
        'amt_cv',
        'vol_ratio_vs_ma5',      # 放量+涨价=积极信号
    ]

    # 处理inf和截尾
    for sf in sub_factors:
        daily[sf] = daily[sf].replace([np.inf, -np.inf], np.nan)
        q1, q99 = daily[sf].quantile(0.01), daily[sf].quantile(0.99)
        daily[sf] = daily[sf].clip(q1, q99)

    # 上影线反转方向：长上影线=抛压重=负信号
    daily['upper_shadow'] = -daily['upper_shadow']

    before = len(daily)
    daily = daily.dropna(subset=sub_factors)
    logger.info(f"  -> 截尾后有效行: {len(daily)}/{before}")

    # 时序z-score
    daily = daily.sort_values(['instrument', 'trading_day']).reset_index(drop=True)
    for sf in sub_factors:
        roll_mean = daily.groupby('instrument')[sf].rolling(20, min_periods=5).mean()
        roll_std = daily.groupby('instrument')[sf].rolling(20, min_periods=5).std()
        daily[f'{sf}_z'] = ((daily[sf] - roll_mean.values) / (roll_std.values + eps))

    # 等权合成
    z_cols = [f'{sf}_z' for sf in sub_factors]
    daily['factor'] = -daily[z_cols].mean(axis=1)
    logger.info(f"  -> 合成完成, 有效因子 {(daily['factor']!=0).sum()}, 耗时 {time.time()-t3:.1f}s")

    # 只保留目标区间数据（前面多取的缓冲期只用于算滚动特征，参考baseline模板）
    daily = daily[(daily['trading_day'] >= pd.to_datetime(start_date)) & (daily['trading_day'] <= pd.to_datetime(end_date))]
    # ============================================================
    # 第4步：输出
    # ============================================================
    logger.info("Step 4: 对齐成分股并输出")
    t4 = time.time()

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [lookback_start, end_date]},
    ).df()
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])

    result = daily.rename(columns={'trading_day': 'date'})
    result = pd.merge(
        result[['date', 'instrument', 'factor']],
        stk_pool,
        how='inner', on=['date', 'instrument']
    )

    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=['factor']).reset_index(drop=True)[['date', 'instrument', 'factor']]

    logger.info(f"  -> 对齐后: {len(result)} 行, {result['date'].nunique()} 个交易日, "
                f"{result['instrument'].nunique()} 只标的, 耗时 {time.time()-t4:.1f}s")
    logger.info(f"因子构建完成! 总耗时 {time.time()-t_total:.1f}s")
    return result


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [lookback_start, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )